# 03 — Detecção de regimes via HMM

Especificação do pré-projeto (seção 3.3): HMM gaussiano (covariância cheia)
sobre observações multivariadas — **retorno agregado da carteira +
volatilidade realizada** —, 2-4 estados via BIC/AIC + inspeção econômica,
Baum-Welch com múltiplas inicializações, Viterbi (caracterização in-sample),
Forward (inferência prospectiva, sem lookahead), robustez via bootstrap em
blocos.

**Este é o fit único/exploratório da Semana 4** (in-sample, para
caracterizar regimes) — o refit causal walk-forward de verdade, que gera a
feature que efetivamente alimenta o LSTM sem vazar informação futura para
os parâmetros do modelo, é escopo da Semana 6.

Duas especificações de observação, ambas do retorno agregado equal-weighted
(já convertido para BRL — ver notebook 02):
- **Principal (3D)**: `return_1d`, `realized_vol_21d`, `realized_vol_63d`
- **Robustez/parcimoniosa (2D)**: `return_1d`, `realized_vol_21d`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.hmm import (
    block_bootstrap_stability,
    characterize_regimes,
    fit_gaussian_hmm,
    forward_filter,
    select_n_states,
    viterbi_decode,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

portfolio_full = pd.read_parquet(PROCESSED_DIR / "portfolio_return.parquet")
valid = portfolio_full.notna().all(axis=1)
groups = valid.ne(valid.shift()).cumsum()
lengths = valid[valid].groupby(groups[valid]).size().tolist()
portfolio = portfolio_full.loc[valid]
print("Sequ?ncias cont?guas:", lengths)
portfolio.tail()

Sequ?ncias cont?guas: [207, 3989]


,return_1d,realized_vol_21d,realized_vol_63d
2024-12-20,-0.002995,0.180696,0.143490
2024-12-23,-0.016488,0.180925,0.146863
2024-12-26,0.006471,0.181341,0.146824
2024-12-27,-0.011832,0.188284,0.148794
2024-12-30,0.003306,0.185349,0.148803


## Especificação principal (3D): retorno + vol. realizada 21d + 63d

In [2]:
cols_3d = ["return_1d", "realized_vol_21d", "realized_vol_63d"]
obs_3d = portfolio[cols_3d].values

selection_3d = select_n_states(obs_3d, state_range=(2, 3, 4), n_init=20, random_state=42, lengths=lengths)
selection_3d

Model is not converging.  Current: 31355.467533611863 is not greater than 31355.485137483232. Delta is -0.0176038713689195


Model is not converging.  Current: 31469.08536361228 is not greater than 31469.15412130289. Delta is -0.06875769060934545


Model is not converging.  Current: 31476.73832423833 is not greater than 31476.743019293226. Delta is -0.004695054896728834


Model is not converging.  Current: 31356.215813652427 is not greater than 31356.221097984304. Delta is -0.005284331877192017


,log_likelihood,bic,aic
n_states,,,
2,29219.635372,-58264.091117,-58397.270743
3,31479.377157,-62666.788270,-62888.754314
4,33131.636134,-65837.836032,-66161.272267


In [3]:
n_states_3d = selection_3d["bic"].idxmin()
print(f"BIC seleciona {n_states_3d} estados (AIC sugere {selection_3d['aic'].idxmin()})")

model_3d = fit_gaussian_hmm(obs_3d, n_states=n_states_3d, n_init=30, random_state=42, lengths=lengths)
regime_summary_3d = characterize_regimes(model_3d, obs_3d, cols_3d)
regime_summary_3d

Model is not converging.  Current: 31469.08536361228 is not greater than 31469.15412130289. Delta is -0.06875769060934545


BIC seleciona 4 estados (AIC sugere 4)


Model is not converging.  Current: 31476.73832423833 is not greater than 31476.743019293226. Delta is -0.004695054896728834


Model is not converging.  Current: 31356.215813652427 is not greater than 31356.221097984304. Delta is -0.005284331877192017


,persistence,expected_duration_days,mean_return_1d,mean_realized_vol_21d,mean_realized_vol_63d
state_rank,,,,,
0,0.990494,105.199065,0.000608,0.140896,0.142524
1,0.977926,45.302600,0.000795,0.172360,0.176437
2,0.984234,63.428532,0.000549,0.216017,0.222614
3,0.976255,42.113524,0.000833,0.395600,0.465238


## Viterbi (in-sample) vs. Forward (prospectivo, sem lookahead)

In [4]:
viterbi_states = viterbi_decode(model_3d, obs_3d, lengths=lengths)
forward_probs = forward_filter(model_3d, obs_3d, lengths=lengths)
forward_state = forward_probs.argmax(axis=1)

regimes = pd.DataFrame(
    {
        "viterbi_state": viterbi_states,
        "forward_state": forward_state,
        **{f"forward_prob_state_{s}": forward_probs[:, s] for s in range(n_states_3d)},
    },
    index=portfolio.index,
)

agreement = (regimes["viterbi_state"] == regimes["forward_state"]).mean()
print(f"Viterbi (in-sample) concorda com Forward (prospectivo) em {agreement:.1%} dos dias")
print("(divergências esperadas nas transições de regime — Viterbi vê o futuro próximo, Forward não)")

regimes.to_parquet(PROCESSED_DIR / "hmm_regimes_insample.parquet")
regimes.tail()

Viterbi (in-sample) concorda com Forward (prospectivo) em 96.6% dos dias
(divergências esperadas nas transições de regime — Viterbi vê o futuro próximo, Forward não)


,viterbi_state,forward_state,forward_prob_state_0,forward_prob_state_1,forward_prob_state_2,forward_prob_state_3
2024-12-20,0,0,0.999952,0.000048,2.528209e-09,7.219389e-07
2024-12-23,0,0,0.999685,0.000312,5.172042e-09,3.241477e-06
2024-12-26,0,0,0.999891,0.000108,1.020482e-08,6.031642e-07
2024-12-27,0,0,0.999434,0.000563,1.557580e-08,2.808260e-06
2024-12-30,0,0,0.999740,0.000259,3.005989e-08,7.572633e-07


## Robustez: bootstrap em blocos

Avalia estabilidade do nº de estados escolhido por BIC e da sequência de
regimes (Viterbi), com estados alinhados por volatilidade entre reamostragens
e comparados via % de concordância + Adjusted Rand Index.

In [5]:
stability_3d = block_bootstrap_stability(
    obs_3d, state_range=(2, 3, 4), block_length=21, n_bootstrap=50, n_init=5, random_state=42, lengths=lengths
)
stability_3d

Model is not converging.  Current: 31355.467533609262 is not greater than 31355.485137481966. Delta is -0.01760387270405772


Model is not converging.  Current: 31469.08536361228 is not greater than 31469.15412130289. Delta is -0.06875769060934545


Model is not converging.  Current: 31469.08536361228 is not greater than 31469.15412130289. Delta is -0.06875769060934545


Model is not converging.  Current: 31860.198621320287 is not greater than 31860.218949119917. Delta is -0.020327799629740184


Model is not converging.  Current: 29197.223516991635 is not greater than 29197.229343320778. Delta is -0.005826329143019393


...

Model is not converging.  Current: 30413.1410879313 is not greater than 30413.14719551425. Delta is -0.006107582950789947


Model is not converging.  Current: 33224.342679423455 is not greater than 33224.355077221604. Delta is -0.012397798149322625


Model is not converging.  Current: 33224.342679423455 is not greater than 33224.355077221604. Delta is -0.012397798149322625


Model is not converging.  Current: 31580.828516875066 is not greater than 31580.860645853154. Delta is -0.03212897808771231


Model is not converging.  Current: 31873.919563399173 is not greater than 31873.956918299766. Delta is -0.03735490059261792


## Especificação de robustez (2D): só retorno + vol. realizada 21d

Versão mais parcimoniosa — `realized_vol_21d` e `realized_vol_63d` são
fortemente correlacionadas (Semana 3), então o modelo 3D pode estar
efetivamente perto de um 2D com peso redobrado na vol. de curto prazo.
Comparar os regimes das duas especificações é um teste de robustez barato.

In [6]:
cols_2d = ["return_1d", "realized_vol_21d"]
obs_2d = portfolio[cols_2d].values

selection_2d = select_n_states(obs_2d, state_range=(2, 3, 4), n_init=20, random_state=42, lengths=lengths)
n_states_2d = selection_2d["bic"].idxmin()
model_2d = fit_gaussian_hmm(obs_2d, n_states=n_states_2d, n_init=30, random_state=42, lengths=lengths)

viterbi_2d = viterbi_decode(model_2d, obs_2d, lengths=lengths)
agreement_2d_vs_3d = (viterbi_2d == viterbi_states).mean() if n_states_2d == n_states_3d else None

print(f"2D seleciona {n_states_2d} estados (3D selecionou {n_states_3d})")
if agreement_2d_vs_3d is not None:
    print(f"Concordância Viterbi 2D vs. 3D: {agreement_2d_vs_3d:.1%}")
else:
    print("Nº de estados difere entre 2D e 3D — comparação direta de rótulo não é direta.")

characterize_regimes(model_2d, obs_2d, cols_2d)

Model is not converging.  Current: 21682.62918182587 is not greater than 21682.655252616663. Delta is -0.026070790794619825


Model is not converging.  Current: 21865.52467360691 is not greater than 21865.52579421149. Delta is -0.001120604581956286


Model is not converging.  Current: 23018.616774595772 is not greater than 23018.631280985202. Delta is -0.014506389430607669


Model is not converging.  Current: 23028.87476564705 is not greater than 23028.874897648206. Delta is -0.00013200115427025594


Model is not converging.  Current: 23011.372263322894 is not greater than 23011.382815754874. Delta is -0.010552431980613619


...

Model is not converging.  Current: 23029.052687805935 is not greater than 23029.057743694888. Delta is -0.005055888952483656


Model is not converging.  Current: 23029.023113601063 is not greater than 23029.028281381557. Delta is -0.005167780494957697


Model is not converging.  Current: 23007.492884585306 is not greater than 23007.504627923205. Delta is -0.011743337898224127


2D seleciona 4 estados (3D selecionou 4)
Concordância Viterbi 2D vs. 3D: 56.6%


,persistence,expected_duration_days,mean_return_1d,mean_realized_vol_21d
state_rank,,,,
0,0.957819,23.707622,0.000581,0.127757
1,0.937459,15.989501,0.000762,0.167131
2,0.952632,21.111510,0.000341,0.216798
3,0.958453,24.069236,0.001628,0.343578


## Salvar relatórios

In [7]:
selection_3d.to_csv(TABLES_DIR / "hmm_state_selection_3d.csv")
regime_summary_3d.to_csv(TABLES_DIR / "hmm_regime_characterization_3d.csv")

pd.Series(stability_3d).drop("selected_n_states_distribution").to_csv(TABLES_DIR / "hmm_bootstrap_stability_3d.csv")
stability_3d["selected_n_states_distribution"].to_csv(TABLES_DIR / "hmm_bootstrap_n_states_distribution_3d.csv")

print("Relatórios salvos em", TABLES_DIR)
print("Regimes (Viterbi + Forward) salvos em", PROCESSED_DIR / "hmm_regimes_insample.parquet")

Relatórios salvos em C:\Python\hmm_lstm_TCC\reports\tables
Regimes (Viterbi + Forward) salvos em C:\Python\hmm_lstm_TCC\data\processed\hmm_regimes_insample.parquet
